In [ ]:
# TimesFM 2.5 + PEFT/LoRA + W&B
!pip -q install timesfm peft wandb pyarrow accelerate
import torch, numpy as np, pandas as pd, os, math, json, wandb
print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")


In [ ]:
import wandb
wandb.login()          # paste your API key when prompted
WANDB_PROJECT = "timesfm-taxi-ft"


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DATA_PATH   = "/content/drive/MyDrive/Timesfm_Chronos_fine-tune/data/taxi_series.parquet"
CKPT_DIR    = "/content/drive/MyDrive/Timesfm_Chronos_fine-tune/timesfm_ft"
RESOLUTIONS = ["1h"]          # start small; add "1d" etc. later. [] = all 9
os.makedirs(CKPT_DIR, exist_ok=True)

df = pd.read_parquet(DATA_PATH)
if RESOLUTIONS:
    df = df[df.resolution.isin(RESOLUTIONS)].copy()
df["ts"] = pd.to_datetime(df["ts"])
df = df.sort_values(["series_id", "split", "ts"]).reset_index(drop=True)
print("rows:", f"{len(df):,}", "| series:", df.series_id.nunique(),
      "| resolutions:", RESOLUTIONS or "ALL")
display(df.groupby(["resolution","split"]).series_id.nunique().unstack(fill_value=0))


In [ ]:
def to_series_dict(frame, split):
    out = {}
    for sid, g in frame[frame.split == split].groupby("series_id"):
        out[sid] = g.sort_values("ts")["value"].to_numpy(dtype=np.float32)
    return out

TRAIN = to_series_dict(df, "train")   # 2015-16, model may learn from this
VAL   = to_series_dict(df, "val")     # 2017 holdout, evaluation only
META  = (df[["series_id","metric","resolution","vendor"]]
         .drop_duplicates().set_index("series_id"))
print("train series:", len(TRAIN), "| val series:", len(VAL))
# length distribution per resolution (drives context_length choice)
lens = pd.DataFrame({"series_id": list(TRAIN), "len": [len(v) for v in TRAIN.values()]})
lens = lens.merge(META, left_on="series_id", right_index=True)
display(lens.groupby("resolution")["len"].describe()[["count","min","50%","max"]])


In [ ]:
class WindowDS(torch.utils.data.Dataset):
    def __init__(self, series_dict, C, H, stride, max_per_series=2000):
        self.items = []   # (sid, start)
        self.data = series_dict
        self.C, self.H = C, H
        for sid, arr in series_dict.items():
            n = len(arr)
            if n < C + H:
                continue
            starts = list(range(0, n - C - H + 1, stride))
            if len(starts) > max_per_series:      # subsample long series
                idx = np.linspace(0, len(starts)-1, max_per_series).astype(int)
                starts = [starts[i] for i in idx]
            self.items += [(sid, s) for s in starts]
    def __len__(self): return len(self.items)
    def __getitem__(self, i):
        sid, s = self.items[i]
        a = self.data[sid]
        ctx = a[s : s+self.C]
        tgt = a[s+self.C : s+self.C+self.H]
        return (torch.from_numpy(ctx.copy()), torch.from_numpy(tgt.copy()))


In [ ]:
def wape(y, p):
    y, p = np.asarray(y,float), np.asarray(p,float)
    d = np.abs(y).sum()
    return float(np.abs(y-p).sum()/d*100) if d else float("nan")

def mase(y, p):
    y, p = np.asarray(y,float), np.asarray(p,float)
    mae = np.abs(y-p).mean()
    naive = np.abs(np.diff(y)).mean() if len(y) > 1 else np.nan
    return float(mae/naive) if naive else float("nan")

MAX_CONTEXT, MAX_HORIZON = 1024, 256


In [ ]:
import timesfm
def load_timesfm(compile_cfg=True):
    m = timesfm.TimesFM_2p5_200M_torch.from_pretrained(
        "google/timesfm-2.5-200m-pytorch", torch_compile=False)
    if compile_cfg:
        m.compile(timesfm.ForecastConfig(
            max_context=MAX_CONTEXT, max_horizon=MAX_HORIZON,
            normalize_inputs=True, use_continuous_quantile_head=True,
            infer_is_positive=True, fix_quantile_crossing=True))
    return m

base = load_timesfm()


In [ ]:
def eval_holdout(model, series_dict, C, H, stride=None, tag="model"):
    stride = stride or H
    rows = []
    for sid, arr in series_dict.items():
        n = len(arr)
        if n < C + H:
            continue
        ys, ps = [], []
        for s in range(0, n - C - H + 1, stride):
            ctx = arr[s:s+C]
            pt, _ = model.forecast(horizon=H, inputs=[ctx])
            ys.append(arr[s+C:s+C+H]); ps.append(np.asarray(pt[0])[:H])
        if not ys:
            continue
        y = np.concatenate(ys); p = np.concatenate(ps)
        r = META.loc[sid]
        rows.append(dict(series_id=sid, resolution=r.resolution, metric=r.metric,
                         vendor=r.vendor, tag=tag, wape=wape(y,p), mase=mase(y,p), n=len(y)))
    return pd.DataFrame(rows)

C_EVAL, H_EVAL = 512, 24
base_res = eval_holdout(base, VAL, C_EVAL, H_EVAL, tag="zero_shot")
print("zero-shot median WAPE:", round(base_res.wape.median(),2),
      "| median MASE:", round(base_res.mase.median(),2))
display(base_res.groupby("resolution")[["wape","mase"]].median())


In [ ]:
# Locate the underlying nn.Module and its Linear layers (LoRA targets)
core = None
for attr in ("model", "_model", "module", "net"):
    if hasattr(base, attr) and isinstance(getattr(base, attr), torch.nn.Module):
        core = getattr(base, attr); break
print("core module found:", type(core).__name__ if core else None)
if core is not None:
    lin = sorted({name.split('.')[-1] for name,m in core.named_modules()
                  if isinstance(m, torch.nn.Linear)})
    print("Linear leaf names (candidate LoRA target_modules):", lin)
    # e.g. attention q/k/v/o projections + MLP; pick from the printed names below


In [ ]:
# Best-effort LoRA training loop — ALIGN forward/loss with the official example.
from peft import LoraConfig, get_peft_model

def build_lora(core, r, alpha, dropout, targets):
    cfg = LoraConfig(r=r, lora_alpha=alpha, lora_dropout=dropout,
                     target_modules=targets, bias="none")
    return get_peft_model(core, cfg)

def train_one(config, log_wandb=True):
    """Fine-tune with a given hyperparameter config; returns val median WAPE."""
    C, H = config["context_length"], config["horizon"]
    model = load_timesfm(compile_cfg=False)              # fresh base each run
    core = None
    for attr in ("model","_model","module","net"):
        if hasattr(model, attr) and isinstance(getattr(model, attr), torch.nn.Module):
            core = getattr(model, attr); break
    assert core is not None, "adjust attr lookup to the installed timesfm layout"

    peft_core = build_lora(core, config["lora_r"], 2*config["lora_r"],
                           config.get("lora_dropout",0.05), config["target_modules"])
    peft_core.train().cuda()

    ds = WindowDS(TRAIN, C, H, stride=config.get("stride", H))
    dl = torch.utils.data.DataLoader(ds, batch_size=config["batch_size"],
                                     shuffle=True, drop_last=True, num_workers=2)
    opt = torch.optim.AdamW([p for p in peft_core.parameters() if p.requires_grad],
                            lr=config["learning_rate"], weight_decay=config["weight_decay"])
    total_steps = config["max_steps"]
    sched = torch.optim.lr_scheduler.OneCycleLR(
        opt, max_lr=config["learning_rate"], total_steps=total_steps,
        pct_start=config.get("warmup_ratio",0.1))

    step = 0
    for ctx, tgt in dl:
        ctx, tgt = ctx.cuda(), tgt.cuda()
        # ── TODO: replace with the official finetuning forward+loss ──────────────
        # The exact call to get a *differentiable* H-step point forecast from `core`
        # (e.g. patched-decoder forward / quantile loss) is version-specific.
        # pred = core_forward(peft_core, ctx, H)                # (B, H)
        # loss = torch.nn.functional.l1_loss(pred, tgt)         # or the repo's quantile loss
        raise NotImplementedError(
            "Wire this to examples/finetuning/ — paste that script and I'll fill it in.")
        # opt.zero_grad(); loss.backward(); opt.step(); sched.step()
        # if log_wandb: wandb.log({"train/loss": loss.item(), "lr": sched.get_last_lr()[0]}, step=step)
        # step += 1
        # if step >= total_steps: break

    # evaluate on holdout with the public API (adapters are active on `model`)
    res = eval_holdout(model, VAL, C, H, tag="ft")
    val_wape = float(res.wape.median())
    if log_wandb: wandb.log({"val_wape": val_wape, "val_mase": float(res.mase.median())})
    # save adapter
    peft_core.save_pretrained(os.path.join(CKPT_DIR, wandb.run.name if wandb.run else "run"))
    return val_wape


In [ ]:
sweep_config = {
    "method": "bayes",
    "metric": {"name": "val_wape", "goal": "minimize"},
    "parameters": {
        "learning_rate": {"distribution": "log_uniform_values", "min": 1e-5, "max": 5e-4},
        "lora_r":        {"values": [8, 16, 32]},
        "context_length":{"values": [336, 672]},
        "horizon":       {"value": 24},
        "weight_decay":  {"values": [0.0, 0.01, 0.1]},
        "batch_size":    {"value": 64},          # A100 can go higher; keep fixed in sweep
        "max_steps":     {"value": 1500},
        "warmup_ratio":  {"value": 0.1},
        "lora_dropout":  {"value": 0.05},
        # target_modules: fill from §3's printed Linear names, e.g. ["q_proj","v_proj"]
        "target_modules":{"value": ["q_proj", "v_proj"]},
    },
    "early_terminate": {"type": "hyperband", "min_iter": 3},
}

def sweep_run():
    with wandb.init(project=WANDB_PROJECT) as run:
        cfg = dict(run.config)
        train_one(cfg, log_wandb=True)

# sweep_id = wandb.sweep(sweep_config, project=WANDB_PROJECT)
# wandb.agent(sweep_id, function=sweep_run, count=20)     # 20 runs on A100


In [ ]:
!git clone https://github.com/google-research/timesfm
!ls timesfm/examples/finetuning/
!cat timesfm/examples/finetuning/*.py

In [ ]:
!find timesfm -type d -iname "*finetun*"
!find timesfm -type f -iname "*finetun*"
!ls timesfm/timesfm-forecasting/examples/

In [ ]:
!ls timesfm/timesfm-forecasting/examples/finetuning/
!cat timesfm/timesfm-forecasting/examples/finetuning/finetune_lora.py

In [ ]:
!pip -q install -U transformers accelerate peft wandb pyarrow
import os, math, numpy as np, pandas as pd, torch
print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")


In [ ]:
import wandb
wandb.login()                       # or comment out; training still runs
os.environ.setdefault("WANDB_PROJECT", "timesfm-taxi-ft")


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DATA_PATH   = "/content/drive/MyDrive/Timesfm_Chronos_fine-tune/data/taxi_series.parquet"
CKPT_DIR    = "/content/drive/MyDrive/Timesfm_Chronos_fine-tune/timesfm_ft_1h"
RESOLUTIONS = []                 # start here; multiple of 32 context needed per res
os.makedirs(CKPT_DIR, exist_ok=True)

df = pd.read_parquet(DATA_PATH)
if RESOLUTIONS:
    df = df[df.resolution.isin(RESOLUTIONS)].copy()
df["ts"] = pd.to_datetime(df["ts"])
df = df.sort_values(["series_id", "split", "ts"]).reset_index(drop=True)

def to_dict(split):
    return {sid: g.sort_values("ts")["value"].to_numpy(np.float32)
            for sid, g in df[df.split == split].groupby("series_id")}
TRAIN, VAL = to_dict("train"), to_dict("val")
META = df[["series_id","metric","resolution","vendor"]].drop_duplicates().set_index("series_id")
# full continuous series + index where val begins (for rolling holdout eval)
FULL = {sid: np.concatenate([TRAIN[sid], VAL.get(sid, np.array([], np.float32))]) for sid in TRAIN}
VAL_START = {sid: len(TRAIN[sid]) for sid in TRAIN}
print("series:", len(TRAIN), "| train pts/series ~", int(np.median([len(v) for v in TRAIN.values()])),
      "| val pts/series ~", int(np.median([len(v) for v in VAL.values()])))


In [ ]:
CONTEXT = 512     # MUST be a multiple of 32 (try 1024 to give 1m more history)
HORIZON = 24
EVAL_MAX_WINDOWS = 150   # per-series cap on rolling holdout windows → keeps all-res eval fast
assert CONTEXT % 32 == 0, "context_len must be a multiple of 32"
MODEL_ID = "google/timesfm-2.5-200m-transformers"

def wape(y, p):
    y, p = np.asarray(y, float), np.asarray(p, float)
    d = np.abs(y).sum(); return float(np.abs(y-p).sum()/d*100) if d else float("nan")
def mase(y, p):
    y, p = np.asarray(y, float), np.asarray(p, float)
    mae = np.abs(y-p).mean(); naive = np.abs(np.diff(y)).mean() if len(y) > 1 else np.nan
    return float(mae/naive) if naive else float("nan")

class RandomWindowDS(torch.utils.data.Dataset):
    "Random full-context windows from the TRAIN arrays (targets always < cutoff → no leakage)."
    def __init__(self, series_list, C, H, num_samples, seed=42):
        self.series_list, self.C, self.H = series_list, C, H
        rng = np.random.default_rng(seed); mn = C + H
        valid = [i for i, s in enumerate(series_list) if len(s) >= mn]
        assert valid, "no TRAIN series long enough for CONTEXT+HORIZON"
        self.samples = []
        for _ in range(num_samples):
            i = int(rng.choice(valid)); s = series_list[i]
            st = int(rng.integers(0, len(s) - mn + 1)); self.samples.append((i, st))
    def __len__(self): return len(self.samples)
    def __getitem__(self, k):
        i, st = self.samples[k]; s = self.series_list[i]
        return (torch.tensor(s[st:st+self.C], dtype=torch.float32),
                torch.tensor(s[st+self.C:st+self.C+self.H], dtype=torch.float32))


In [ ]:
!pip uninstall -y torchao

In [ ]:
from transformers import TimesFm2_5ModelForPrediction
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def load_model():
    return TimesFm2_5ModelForPrediction.from_pretrained(
        MODEL_ID, torch_dtype=torch.bfloat16, device_map=DEVICE).eval()

@torch.no_grad()
def eval_holdout(model, full, val_start, meta, C, H, tag="model", batch=128,
                 max_windows=EVAL_MAX_WINDOWS):
    windows = []
    for sid, arr in full.items():
        vs = val_start[sid]
        origins = [o for o in range(vs, len(arr) - H + 1, H) if o - C >= 0]
        if max_windows and len(origins) > max_windows:      # even subsample (fast 1m/3m eval)
            origins = [origins[i] for i in np.linspace(0, len(origins)-1, max_windows).astype(int)]
        for o in origins:
            windows.append((sid, arr[o-C:o], arr[o:o+H]))
    per = {}
    for i in range(0, len(windows), batch):
        chunk = windows[i:i+batch]
        X = torch.tensor(np.stack([c for _, c, _ in chunk]), dtype=torch.float32, device=DEVICE)
        mp = model(past_values=X).mean_predictions[:, :H].float().cpu().numpy()
        for j, (sid, _, tgt) in enumerate(chunk):
            per.setdefault(sid, ([], [])); per[sid][0].append(tgt); per[sid][1].append(mp[j])
    rows = []
    for sid, (ys, ps) in per.items():
        y = np.concatenate(ys); p = np.concatenate(ps); r = meta.loc[sid]
        rows.append(dict(series_id=sid, metric=r.metric, resolution=r.resolution,
                         vendor=r.vendor, tag=tag, wape=wape(y, p), mase=mase(y, p), n=len(y)))
    return pd.DataFrame(rows)

base = load_model()
base_res = eval_holdout(base, FULL, VAL_START, META, CONTEXT, HORIZON, tag="zero_shot")
print("ZERO-SHOT  median WAPE:", round(base_res.wape.median(), 2),
      "| median MASE:", round(base_res.mase.median(), 2))
display(base_res.groupby("resolution")[["wape", "mase"]].median())


In [ ]:
from peft import LoraConfig, get_peft_model, PeftModel

def train_ft(lr=1e-4, lora_r=8, lora_alpha=16, lora_dropout=0.05,
             epochs=3, batch_size=16, num_samples=12000, log_wandb=True):
    if log_wandb:
        wandb.init(project=os.environ["WANDB_PROJECT"],
                   config=dict(lr=lr, lora_r=lora_r, lora_alpha=lora_alpha,
                               epochs=epochs, batch_size=batch_size,
                               context=CONTEXT, horizon=HORIZON, num_samples=num_samples))
    model = get_peft_model(load_model(), LoraConfig(
        r=lora_r, lora_alpha=lora_alpha, target_modules="all-linear",
        lora_dropout=lora_dropout, bias="none"))
    model.print_trainable_parameters(); model.train()

    ds = RandomWindowDS(list(TRAIN.values()), CONTEXT, HORIZON, num_samples)
    dl = torch.utils.data.DataLoader(ds, batch_size=batch_size, shuffle=True, drop_last=True)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs*len(dl))

    step = 0
    for ep in range(1, epochs+1):
        model.train()
        for ctx, tgt in dl:
            ctx, tgt = ctx.to(DEVICE), tgt.to(DEVICE)
            out = model(past_values=ctx, future_values=tgt, forecast_context_len=CONTEXT)
            out.loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); opt.zero_grad(); sched.step(); step += 1
            if log_wandb and step % 20 == 0:
                wandb.log({"train/loss": out.loss.item(), "lr": sched.get_last_lr()[0]}, step=step)
        model.eval()
        r = eval_holdout(model, FULL, VAL_START, META, CONTEXT, HORIZON, tag="ft")
        vw, vm = float(r.wape.median()), float(r.mase.median())
        print(f"epoch {ep}: holdout median WAPE {vw:.2f} | MASE {vm:.2f}")
        if log_wandb: wandb.log({"val_wape": vw, "val_mase": vm, "epoch": ep})
    model.save_pretrained(CKPT_DIR)
    if log_wandb: wandb.finish()
    print("saved adapter →", CKPT_DIR)
    return model, r

ft_model, ft_res = train_ft()


In [ ]:
cmp = (pd.concat([base_res.assign(tag="zero_shot"), ft_res.assign(tag="ft")])
         .groupby(["resolution", "tag"])[["wape", "mase"]].median().unstack())
display(cmp)
print("overall  zero-shot WAPE %.2f / MASE %.2f  →  ft WAPE %.2f / MASE %.2f" % (
    base_res.wape.median(), base_res.mase.median(), ft_res.wape.median(), ft_res.mase.median()))
print("Ship FT only where it beats zero-shot on the holdout. Baseline is already strong (MASE<1).")


In [ ]:
SWEEP = {
  "method": "bayes",
  "metric": {"name": "val_wape", "goal": "minimize"},
  "parameters": {
    "lr":          {"distribution": "log_uniform_values", "min": 2e-5, "max": 5e-4},
    "lora_r":      {"values": [4, 8, 16]},
    "lora_alpha":  {"values": [8, 16, 32]},
    "epochs":      {"value": 3},
    "batch_size":  {"value": 16},
    "num_samples": {"value": 5000},
  },
  "early_terminate": {"type": "hyperband", "min_iter": 1},
}
def sweep_run():
    with wandb.init() as run:
        c = run.config
        train_ft(lr=c.lr, lora_r=c.lora_r, lora_alpha=c.lora_alpha,
                 epochs=c.epochs, batch_size=c.batch_size,
                 num_samples=c.num_samples, log_wandb=True)
# sweep_id = wandb.sweep(SWEEP, project=os.environ["WANDB_PROJECT"])
# wandb.agent(sweep_id, function=sweep_run, count=15)
